# NLP Homework — Solution (Week 1, Day 1)

Reference: `nlp_homework.md`.

The implementation lives in the **`nlp_homework/`** package next to this notebook,
not in the notebook itself. This keeps every function unit-testable and lets the
notebook stay a presentation layer.

| Module | Covers |
|---|---|
| `nlp_homework.dataset` | Synthetic corpus generation |
| `nlp_homework.preprocessing` | Task 1 — cleaning, tokenising, stemming, lemmatising |
| `nlp_homework.exploration` | Task 2 — statistics, n-grams, TF-IDF, figures |
| `nlp_homework.ner` | Task 3 — NER, evaluation, entity highlighting |
| `nlp_homework.pipeline` | End-to-end orchestration |

Run the test suite from the `Week1/` directory:

```bash
uv run pytest Day_1/tests -v
```

## Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

sys.path.insert(0, str(Path.cwd()))

from nlp_homework import dataset, exploration, ner, pipeline, preprocessing

pd.set_option('display.max_colwidth', 90)

# Download NLTK corpora (no-op once cached).
preprocessing.ensure_nltk_resources()
print('Ready.')

## Dataset

Generated with an injected `random.Random(seed)` rather than the global RNG, and
shuffled with an explicit `random_state`, so the corpus is genuinely reproducible.

In [ ]:
reviews_df = dataset.generate_dataset(n_reviews=1000, seed=42)
dataset.save_dataset(reviews_df, 'movie_reviews.csv')

print(f'{len(reviews_df)} reviews  |  class balance: '
      f'{reviews_df["sentiment"].value_counts().to_dict()}')
reviews_df.head()

---
## Task 1 — Text Preprocessing

Two corrections against the naive approach:

1. **Tokenise before stripping punctuation.** Stripping first turns `don't` into
   `dont` — no longer a stopword, so it survives as noise — and collapses
   `soc.religion.christian` into a single nonsense token.
2. **Lemmatise with a POS tag.** `WordNetLemmatizer` defaults to noun, so an
   untagged call leaves `running` as `running` and `better` as `better`.

In [ ]:
sample_review = reviews_df.iloc[0]['review']
print('ORIGINAL:\n', sample_review[:400], '...\n')

processed = preprocessing.preprocess(sample_review)
print('CLEANED:\n', processed.cleaned[:400], '...\n')
print('TOKENS   :', processed.tokens[:12])
print('STEMMED  :', processed.stemmed[:12])
print('LEMMATIZED:', processed.lemmatized[:12])

In [ ]:
# Apply to the whole corpus (one tokenisation pass per review).
task_one = pipeline.run_task_one(reviews_df)
reviews_df = task_one.frame

reviews_df[['review', 'preprocessed', 'word_count']].head(3)

### Stemming vs. lemmatisation

In [ ]:
comparison = pd.DataFrame(task_one.comparison)
display(comparison)

print('\nDiscussion (derived from the table above, not hardcoded):')
for note in task_one.discussion:
    print(' •', note)

**Reading the table.** Stemming truncates by rule, so it produces forms that are
not words (`studi`, `movi`) and cannot resolve irregulars — `better` and `worst`
pass through untouched. POS-aware lemmatisation returns dictionary words
throughout and does resolve the irregulars (`better` → `well`, `worst` → `bad`).

Lemmatisation is the right choice here: Task 3 matches entity surface forms
against the source text, and truncated stems no longer align with it.

---
## Task 2 — Exploration and Visualisation

Computation is separated from plotting, so every statistic below is unit-tested
without needing a display backend.

In [ ]:
task_two = pipeline.run_task_two(reviews_df, output_dir=Path('.'))

stats = task_two.statistics
print(f"Documents        : {stats['n_documents']}")
print(f"Average length   : {stats['average_length']:.2f} words")
print(f"Median length    : {stats['median_length']:.0f} words")
print(f"Length range     : {stats['min_length']}–{stats['max_length']} words")
print(f"Vocabulary size  : {stats['vocabulary_size']:,} unique tokens")
print(f"Total tokens     : {stats['total_tokens']:,}")

In [ ]:
pd.DataFrame({
    'positive': [f'{w} ({c})' for w, c in task_two.positive_words[:15]],
    'negative': [f'{w} ({c})' for w, c in task_two.negative_words[:15]],
}).rename_axis('rank')

In [ ]:
for path in task_two.figures:
    print('•', path)

display(HTML(f'<img src="review_length_distribution.png" width="700">'))
display(HTML('<img src="positive_wordcloud.png" width="700">'))
display(HTML('<img src="negative_wordcloud.png" width="700">'))

In [ ]:
pd.DataFrame({
    'positive bigram': [g for g, _ in task_two.positive_bigrams[:10]],
    'negative bigram': [g for g, _ in task_two.negative_bigrams[:10]],
    'positive trigram': [g for g, _ in task_two.positive_trigrams[:10]],
    'negative trigram': [g for g, _ in task_two.negative_trigrams[:10]],
})

In [ ]:
display(HTML('<img src="bigram_frequencies.png" width="750">'))
display(HTML('<img src="trigram_frequencies.png" width="750">'))
display(HTML('<img src="tfidf_scores.png" width="750">'))

**Interpretation.** The injected polarity vocabulary dominates both the
frequency counts and the TF-IDF rankings, exactly as the generator intends —
`great`/`enjoyable`/`fantastic` on the positive side, `awful`/`waste`/`boring`
on the negative. Because the filler prose is drawn from 20-newsgroups at random,
the n-grams are mostly incidental word pairs rather than stable collocations;
that is a property of the synthetic corpus, not a defect in the counting.

---
## Task 3 — Named Entity Recognition

Entities carry **character offsets** end to end (`Entity(start, end, label, text)`).
Deriving positions lazily with `str.find` at render time — as the naive version
did — causes three separate defects:

* only the *first* mention of each term is ever found;
* `Oscar` matches inside `Oscars`, `Avatar` inside `Avatar-like`;
* overlapping spans (NLTK's `PERSON` vs. our `DIRECTOR`) are emitted twice,
  duplicating text in the highlighted output.

In [ ]:
demo = ("Steven Spielberg's Jurassic Park won an Oscar. "
        "Tom Hanks and Leonardo DiCaprio starred; Tom Hanks was the standout.")

print('NLTK ne_chunk:')
for e in ner.extract_entities_nltk(demo):
    print(f'  [{e.start:3d}:{e.end:3d}] {e.label:12} {e.text}')

print('\nCustom movie NER:')
for e in ner.custom_movie_ner(demo):
    print(f'  [{e.start:3d}:{e.end:3d}] {e.label:12} {e.text}')

print('\nNote both "Tom Hanks" mentions are found, at distinct offsets.')

In [ ]:
# Boundary handling: near-misses must not fire.
for text in ['She collects Oscars memorabilia.',
             'He admired Avatar-like visual design.',
             'Daniel Day-Lewis retired.']:
    print(f'{text:45} -> {[(e.label, e.text) for e in ner.custom_movie_ner(text)]}')

In [ ]:
task_three = pipeline.run_task_three(reviews_df, output_dir=Path('.'), sample_size=50)

print(f'NLTK entities  : {len(task_three.entities)}')
print(f'Custom entities: {len(task_three.custom_entities)}\n')
display(task_three.entities['entity_type'].value_counts().to_frame('count'))
display(task_three.custom_entities['entity_type'].value_counts().to_frame('count'))

In [ ]:
display(HTML('<img src="entity_type_distribution.png" width="650">'))
display(HTML('<img src="top_entities_by_type.png" width="750">'))
display(HTML('<img src="entity_comparison.png" width="650">'))
display(HTML('<img src="custom_entity_types.png" width="650">'))

### Evaluating the custom NER — honestly

The brief's sample test set contains only positive examples drawn from the same
gazetteer that powers the recogniser, so it scores a meaningless **F1 = 1.00**.
The set in `nlp_homework/evaluation_data.py` adds cases a dictionary
*cannot* get right:

* `unseen_entity` — real entities absent from the gazetteer (Christian Bale,
  *The Departed*). These establish the honest recall ceiling.
* `negative` — no entities at all; any prediction is a false positive.
* `substring_trap` — `Oscars`, `Avatar-like`.

In [ ]:
m = task_three.overall_metrics
print(f"Overall  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}"
      f"   (TP={m['true_positives']} FP={m['false_positives']} FN={m['false_negatives']})\n")

display(pd.DataFrame(task_three.category_metrics).T[
    ['precision', 'recall', 'f1', 'true_positives', 'false_positives', 'false_negatives']])
display(pd.DataFrame(task_three.per_label_metrics).T[['precision', 'recall', 'f1']])

**Interpretation.** Precision is 1.00 and there are zero false positives on the
negative and substring-trap cases — the boundary-aware matcher does not
hallucinate. Recall is capped by coverage, not by the algorithm: every miss is an
entity outside the gazetteer, which is the defining limitation of a dictionary
recogniser. `ACTOR` recall is the weakest label for exactly that reason.

The honest conclusion: **this approach cannot generalise beyond its dictionary.**
Closing that gap needs a statistical or transformer-based model, not a longer
word list.

### Entity highlighting (Task 3, item 7)

`highlight_entities` resolves overlapping spans — preferring the more specific
label, so `DIRECTOR` wins over NLTK's `PERSON` — and HTML-escapes both the
entity text and the surrounding text.

In [ ]:
legend = ' '.join(
    f'<span style="background-color:{c}; padding:2px 6px; margin-right:4px;'
    f' border-radius:3px;">{label}</span>'
    for label, c in ner.ENTITY_COLORS.items())
display(HTML(f'<div style="line-height:2.2">{legend}</div>'))

display(HTML(f'<p style="line-height:1.9">'
             f'{ner.highlight_entities(demo, ner.extract_all_entities(demo))}</p>'))

In [ ]:
for review_id, highlighted in task_three.highlighted_samples:
    sentiment = 'Positive' if reviews_df.loc[review_id, 'sentiment'] == 1 else 'Negative'
    display(HTML(f'<h4>Review {review_id} — {sentiment}</h4>'
                 f'<p style="line-height:1.9">{highlighted}</p>'))

---
## Summary

| Task | Deliverable | Where |
|---|---|---|
| 1 | Lowercase, punctuation, numbers, stopwords, whitespace | `preprocessing.tokenize` |
| 1 | Tokenisation | `preprocessing.tokenize` |
| 1 | Stemming + POS-aware lemmatisation | `preprocessing.stem_tokens` / `lemmatize_tokens` |
| 1 | Stem-vs-lemma comparison + discussion | `preprocessing.describe_comparison` |
| 2 | Length statistics + distribution | `exploration.corpus_statistics` |
| 2 | Vocabulary size | `exploration.corpus_statistics` |
| 2 | Common words by sentiment | `exploration.most_common_words` |
| 2 | Word clouds | `exploration.plot_wordcloud` |
| 2 | Bigram / trigram frequencies | `exploration.ngram_frequencies` |
| 2 | TF-IDF top 20 | `exploration.top_tfidf_terms` |
| 3 | NLTK NER over 50 reviews | `ner.extract_entities_nltk` |
| 3 | Entity counts by type and sentiment | `pipeline.run_task_three` |
| 3 | Three entity visualisations | `exploration.plot_*` |
| 3 | Custom movie NER | `ner.custom_movie_ner` |
| 3 | Evaluation on a labelled test set | `ner.evaluate` / `evaluate_by_label` |
| 3 | Colour-coded entity highlighting | `ner.highlight_entities` |

### Findings

1. POS-aware lemmatisation is what makes lemmatisation worth its cost; untagged,
   it barely differs from doing nothing on inflected forms.
2. Character offsets, carried from extraction through to rendering, are what let
   repeated mentions, boundary correctness, and overlap resolution all be handled
   in one place.
3. A gazetteer recogniser is precise but cannot generalise. Evaluating it only on
   in-dictionary examples hides that completely — the F1 goes from a meaningless
   1.00 to an informative ~0.94 the moment unseen entities are added.